In [867]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# Load Data 

In [868]:
data_path = Path("../data")

movie_path = f"{data_path}/refine_movies.csv"
rating_path = f"{data_path}/refine_ratings.csv"
tag_path = f"{data_path}/refine_tags.csv"
link_path = f"{data_path}/refine_links.csv"

In [869]:
movies_df = pd.read_csv(
    movie_path,
    usecols=['movieId', 'title','year','genres'],
    dtype={'title':str, 'year':str, 'movieId':int, 'genres':str})

ratings_df = pd.read_csv(
    rating_path,
    usecols=['userId','movieId','rating'],
    dtype={'userId':int, 'movieId':int, 'rating':float}
    )
tags_df = pd.read_csv(
    tag_path,
    usecols=['userId','movieId','tag'],
    dtype={'userId':int, 'movieId':int, 'tag':str})
links_df = pd.read_csv(link_path)

In [870]:
df_list = {
    "movies" : movies_df,
    "ratings" : ratings_df,
    "tags" : tags_df,
    "links_df" : links_df
}

# Data check

In [871]:
# genres to list
import ast
movies_df['genres'] = movies_df['genres'].apply(lambda x : ast.literal_eval(x))

movies_df['genres']

0       [Adventure, Animation, Children, Comedy, Fantasy]
1                          [Adventure, Children, Fantasy]
2                                       [Comedy, Romance]
3                                [Comedy, Drama, Romance]
4                                                [Comedy]
                              ...                        
9734                 [Action, Animation, Comedy, Fantasy]
9735                         [Animation, Comedy, Fantasy]
9736                                              [Drama]
9737                                  [Action, Animation]
9738                                             [Comedy]
Name: genres, Length: 9739, dtype: object

In [872]:
ratings_df

,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0
...,...,...,...
100831,610,166534,4.0
100832,610,168248,5.0
100833,610,168250,5.0
100834,610,168252,5.0


In [873]:
tags_df

,userId,movieId,tag
0,2,60756,['funny']
1,2,60756,"['Highly', 'quotable']"
2,2,60756,"['will', 'ferrell']"
3,2,89774,"['Boxing', 'story']"
4,2,89774,['MMA']
...,...,...,...
3678,606,7382,"['for', 'katie']"
3679,606,7936,['austere']
3680,610,3265,"['gun', 'fu']"
3681,610,3265,"['heroic', 'bloodshed']"


# Data Merge

## (1) rating centering
: 유저 bias 제거 . 분산도 함께 반영

### Z score normalization

In [874]:
temp_rating = ratings_df.copy()

func = lambda rating : (rating - rating.mean()) / rating.std()

temp_rating['centering_rating'] = ratings_df.groupby(by='userId')['rating'].transform(func) # apply 는 원래 구조를 변경, transform 는 원래 DF 구조를 유지

temp_rating['centering_rating'] = temp_rating['centering_rating'].fillna(0) # 평가가 1개인 경우 std=0 이라서 NaN 값이 출력 → 0 으로 수정

ratings_df = temp_rating

In [875]:
movie_ratings_df = pd.merge(ratings_df, movies_df, on='movieId', how='left') # on : merge key
movie_ratings_df

,userId,movieId,rating,centering_rating,title,genres,year
0,1,1,4.0,-0.457947,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995)
1,1,3,4.0,-0.457947,Grumpier Old Men (1995),"[Comedy, Romance]",(1995)
2,1,6,4.0,-0.457947,Heat (1995),"[Action, Crime, Thriller]",(1995)
3,1,47,5.0,0.791978,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",(1995)
4,1,50,5.0,0.791978,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",(1995)
...,...,...,...,...,...,...,...
100831,610,166534,4.0,0.363233,Split (2017),"[Drama, Horror, Thriller]",(2017)
100832,610,168248,5.0,1.529520,John Wick: Chapter Two (2017),"[Action, Crime, Thriller]",(2017)
100833,610,168250,5.0,1.529520,Get Out (2017),[Horror],(2017)
100834,610,168252,5.0,1.529520,Logan (2017),"[Action, Sci-Fi]",(2017)


# zero ratings movie check

In [876]:
movie_ratings_df.isnull().sum()

userId              0
movieId             0
rating              0
centering_rating    0
title               0
genres              0
year                0
dtype: int64

In [877]:
no_ratings_movieId = movie_ratings_df.loc[movie_ratings_df.isnull().any(axis=1)]['movieId'].values
print("평가가 존재하지 않는 영화 ID 갯수: ",len(no_ratings_movieId))
no_ratings_movieId

평가가 존재하지 않는 영화 ID 갯수:  0


array([], dtype=int64)

In [878]:
movie_ratings_df[movie_ratings_df['movieId'].isin(no_ratings_movieId)]

,userId,movieId,rating,centering_rating,title,genres,year


In [879]:
# double check (ratings_df 에도 존재하지 않는지 확인)
ratings_df[ratings_df['movieId'].isin(no_ratings_movieId)]

,userId,movieId,rating,centering_rating


# Item User Matrix

In [880]:
item_user_matrix = movie_ratings_df.pivot_table(values='centering_rating', index='title', columns='userId')

In [881]:
item_user_matrix

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.363233
'Hellboy': The Seeds of Creation (2004),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Round Midnight (1986),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Salem's Lot (2004),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Til There Was You (1997),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
eXistenZ (1999),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1.29334,NaN,NaN,NaN,NaN,1.265517,NaN,NaN
xXx (2002),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.776678,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.338958,NaN,-1.969341
xXx: State of the Union (2005),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2.552484


## (1) 영화당 평균 평점 갯수 계산

In [882]:
print("평균 평점 갯수 :" ,round(item_user_matrix.apply(lambda x: x.count(), axis=1).mean()))
ratings_count = item_user_matrix.apply(lambda x: x.count(), axis=1)
ratings_count

평균 평점 갯수 : 10


title
'71 (2014)                                    1
'Hellboy': The Seeds of Creation (2004)       1
'Round Midnight (1986)                        2
'Salem's Lot (2004)                           1
'Til There Was You (1997)                     2
                                             ..
eXistenZ (1999)                              22
xXx (2002)                                   24
xXx: State of the Union (2005)                5
¡Three Amigos! (1986)                        26
À nous la liberté (Freedom for Us) (1931)     1
Length: 9721, dtype: int64

In [883]:
ratings_count.sort_values(ascending=False)

title
Forrest Gump (1994)                          329
Shawshank Redemption, The (1994)             317
Pulp Fiction (1994)                          307
Silence of the Lambs, The (1991)             279
Matrix, The (1999)                           278
                                            ... 
King Solomon's Mines (1937)                    1
King Ralph (1991)                              1
King Kong Lives (1986)                         1
Kindred, The (1986)                            1
À nous la liberté (Freedom for Us) (1931)      1
Length: 9721, dtype: int64

In [884]:
movies_df

,movieId,title,genres,year
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995)
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",(1995)
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",(1995)
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",(1995)
4,5,Father of the Bride Part II (1995),[Comedy],(1995)
...,...,...,...,...
9734,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]",(2017)
9735,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]",(2017)
9736,193585,Flint (2017),[Drama],(2017)
9737,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]",(2018)


In [885]:
temp_df = movies_df.set_index(keys='title').copy()
temp_df = temp_df.join(ratings_count.rename('count'))
movies_df = temp_df.reset_index()

movies_df['count'] = movies_df['count']
movies_df['count'] = movies_df['count'].fillna(0)
movies_df['count'] = movies_df['count'].astype('int64')

movies_df

,title,movieId,genres,year,count
0,Toy Story (1995),1,"[Adventure, Animation, Children, Comedy, Fantasy]",(1995),215
1,Jumanji (1995),2,"[Adventure, Children, Fantasy]",(1995),110
2,Grumpier Old Men (1995),3,"[Comedy, Romance]",(1995),52
3,Waiting to Exhale (1995),4,"[Comedy, Drama, Romance]",(1995),7
4,Father of the Bride Part II (1995),5,[Comedy],(1995),49
...,...,...,...,...,...
9734,Black Butler: Book of the Atlantic (2017),193581,"[Action, Animation, Comedy, Fantasy]",(2017),1
9735,No Game No Life: Zero (2017),193583,"[Animation, Comedy, Fantasy]",(2017),1
9736,Flint (2017),193585,[Drama],(2017),1
9737,Bungo Stray Dogs: Dead Apple (2018),193587,"[Action, Animation]",(2018),1


In [886]:
item_user_matrix

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.363233
'Hellboy': The Seeds of Creation (2004),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Round Midnight (1986),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Salem's Lot (2004),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Til There Was You (1997),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
eXistenZ (1999),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1.29334,NaN,NaN,NaN,NaN,1.265517,NaN,NaN
xXx (2002),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.776678,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.338958,NaN,-1.969341
xXx: State of the Union (2005),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2.552484


# 유저별 장르 평균 점수

In [887]:
genres_list = movies_df['genres'].explode().unique()

In [888]:
movies_df

,title,movieId,genres,year,count
0,Toy Story (1995),1,"[Adventure, Animation, Children, Comedy, Fantasy]",(1995),215
1,Jumanji (1995),2,"[Adventure, Children, Fantasy]",(1995),110
2,Grumpier Old Men (1995),3,"[Comedy, Romance]",(1995),52
3,Waiting to Exhale (1995),4,"[Comedy, Drama, Romance]",(1995),7
4,Father of the Bride Part II (1995),5,[Comedy],(1995),49
...,...,...,...,...,...
9734,Black Butler: Book of the Atlantic (2017),193581,"[Action, Animation, Comedy, Fantasy]",(2017),1
9735,No Game No Life: Zero (2017),193583,"[Animation, Comedy, Fantasy]",(2017),1
9736,Flint (2017),193585,[Drama],(2017),1
9737,Bungo Stray Dogs: Dead Apple (2018),193587,"[Action, Animation]",(2018),1


In [889]:
def calc_user_genre_mean(group):
    genres_totals = {genre : 0 for genre in genres_list}
    genres_count = {genre : 0 for genre in genres_list}
    
    for _, row in group.iterrows():
        rating = row['centering_rating']
        movie_genres = row['genres']
        for genre in movie_genres:
            genres_totals[genre] += rating
            genres_count[genre] += 1
    genres_avg = {}
    for genre in genres_list:
        if genres_count[genre] > 0 :
            genres_avg[genre] = genres_totals[genre]/genres_count[genre]
        else:
            genres_avg[genre] = pd.NA
    return pd.Series(genres_avg)

In [890]:
movie_ratings_df

,userId,movieId,rating,centering_rating,title,genres,year
0,1,1,4.0,-0.457947,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",(1995)
1,1,3,4.0,-0.457947,Grumpier Old Men (1995),"[Comedy, Romance]",(1995)
2,1,6,4.0,-0.457947,Heat (1995),"[Action, Crime, Thriller]",(1995)
3,1,47,5.0,0.791978,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",(1995)
4,1,50,5.0,0.791978,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",(1995)
...,...,...,...,...,...,...,...
100831,610,166534,4.0,0.363233,Split (2017),"[Drama, Horror, Thriller]",(2017)
100832,610,168248,5.0,1.529520,John Wick: Chapter Two (2017),"[Action, Crime, Thriller]",(2017)
100833,610,168250,5.0,1.529520,Get Out (2017),[Horror],(2017)
100834,610,168252,5.0,1.529520,Logan (2017),"[Action, Sci-Fi]",(2017)


In [891]:
user_genre_avg = movie_ratings_df.groupby('userId').apply(calc_user_genre_mean)

/var/folders/1t/sts9dw9x317fjpv6tzhvwldw0000gn/T/ipykernel_49603/2273720666.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  user_genre_avg = movie_ratings_df.groupby('userId').apply(calc_user_genre_mean)


In [892]:
user_genre_avg

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir
userId,,,,,,,,,,,,,,,,,,,
1,0.027318,0.404071,0.226536,-0.111582,-0.085629,-0.073354,0.203778,-0.055193,-0.013529,-0.276139,-1.119672,-0.249626,-0.176714,0.167016,0.394275,<NA>,<NA>,-0.100825,0.791978
2,0.271086,<NA>,<NA>,0.064205,<NA>,0.684849,-0.081829,0.007782,-0.184053,-0.308182,-1.177084,0.064205,-0.090956,0.684849,<NA>,0.477967,-0.246118,-0.55644,<NA>
3,0.030662,-0.925982,-0.925982,-0.686821,0.449193,-0.925982,-0.806402,0.54315,-0.925982,0.816476,1.076991,1.226467,0.843809,-0.925982,-0.925982,<NA>,<NA>,<NA>,<NA>
4,0.0758,0.338185,0.186002,-0.034957,0.097896,-0.134108,-0.054955,-0.179238,0.197275,-0.002225,0.528415,-0.058815,-0.549551,0.012078,0.338185,0.338185,-0.422732,0.186002,0.338185
5,-0.390093,0.703697,0.47933,-0.171335,0.511382,-0.550719,0.165216,-0.530322,0.198871,-0.081588,-0.642506,0.367146,-1.147331,-0.305955,0.771007,<NA>,0.030596,-0.642506,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,-0.212669,0.07856,-0.287824,-0.127159,-0.082115,0.114303,0.180310,-0.660928,-0.004507,-0.182668,-0.429825,0.184789,-0.138702,0.186307,0.096494,0.19693,-0.821547,-0.339218,0.214192
607,-0.33079,-0.468865,-0.378026,-0.475141,-0.222302,-0.278416,0.234140,-0.066146,0.02974,0.340346,0.339861,0.891582,-0.555162,0.394105,-0.192715,<NA>,1.257075,0.221511,<NA>
608,0.080443,-0.014819,-0.624453,-0.368359,-0.124322,-0.229215,0.281048,0.181744,0.443672,0.372944,0.171795,0.385957,0.150317,0.412107,-0.348942,-0.124322,0.802237,-0.461252,0.570598


# 영화별 장르 매트릭스 계산

In [893]:
genres_list

array(['Adventure', 'Animation', 'Children', 'Comedy', 'Fantasy',
       'Romance', 'Drama', 'Action', 'Crime', 'Thriller', 'Horror',
       'Mystery', 'Sci-Fi', 'War', 'Musical', 'Documentary', 'IMAX',
       'Western', 'Film-Noir'], dtype=object)

In [894]:
movies_df

,title,movieId,genres,year,count
0,Toy Story (1995),1,"[Adventure, Animation, Children, Comedy, Fantasy]",(1995),215
1,Jumanji (1995),2,"[Adventure, Children, Fantasy]",(1995),110
2,Grumpier Old Men (1995),3,"[Comedy, Romance]",(1995),52
3,Waiting to Exhale (1995),4,"[Comedy, Drama, Romance]",(1995),7
4,Father of the Bride Part II (1995),5,[Comedy],(1995),49
...,...,...,...,...,...
9734,Black Butler: Book of the Atlantic (2017),193581,"[Action, Animation, Comedy, Fantasy]",(2017),1
9735,No Game No Life: Zero (2017),193583,"[Animation, Comedy, Fantasy]",(2017),1
9736,Flint (2017),193585,[Drama],(2017),1
9737,Bungo Stray Dogs: Dead Apple (2018),193587,"[Action, Animation]",(2018),1


In [895]:
movie_genre_matrix = pd.DataFrame(0, index=movies_df['title'], columns=genres_list)
movie_genre_matrix

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir
title,,,,,,,,,,,,,,,,,,,
Toy Story (1995),0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Jumanji (1995),0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Grumpier Old Men (1995),0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Waiting to Exhale (1995),0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Father of the Bride Part II (1995),0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Black Butler: Book of the Atlantic (2017),0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
No Game No Life: Zero (2017),0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Flint (2017),0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [896]:
for genre in genres_list:
    movie_genre_matrix[genre] = movies_df.set_index('title')['genres'].apply(lambda x : 1 if genre in x else 0) # apply 적용을 위해 movies_df 의 인덱스 movieId 로 설정
    
movie_genre_matrix

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir
title,,,,,,,,,,,,,,,,,,,
Toy Story (1995),1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Jumanji (1995),1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Grumpier Old Men (1995),0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
Waiting to Exhale (1995),0,0,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0
Father of the Bride Part II (1995),0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Black Butler: Book of the Atlantic (2017),0,1,0,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0
No Game No Life: Zero (2017),0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Flint (2017),0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0


In [897]:
genre_count = movie_genre_matrix.sum(axis=1)

movie_genre_matrix = movie_genre_matrix.div(genre_count.replace(0,1), axis=0) # 장르가 많은 영화의 점수가 많이 반영되지 않도록 정규화

movie_genre_matrix

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir
title,,,,,,,,,,,,,,,,,,,
Toy Story (1995),0.200000,0.200000,0.200000,0.200000,0.200000,0.000000,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Jumanji (1995),0.333333,0.000000,0.333333,0.000000,0.333333,0.000000,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Grumpier Old Men (1995),0.000000,0.000000,0.000000,0.500000,0.000000,0.500000,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Waiting to Exhale (1995),0.000000,0.000000,0.000000,0.333333,0.000000,0.333333,0.333333,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Father of the Bride Part II (1995),0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Black Butler: Book of the Atlantic (2017),0.000000,0.250000,0.000000,0.250000,0.250000,0.000000,0.000000,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
No Game No Life: Zero (2017),0.000000,0.333333,0.000000,0.333333,0.333333,0.000000,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Flint (2017),0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [898]:
print(user_genre_avg.shape)
print(movie_genre_matrix.shape)

(610, 19)
(9739, 19)


# 예측 평점 행렬 계산

user_genre_avg 와 movie_genre_matrix 내적
- 결과 shape 사용자 수 x 영화 수

In [899]:
pred_ratings_use_genre = user_genre_avg.dot(movie_genre_matrix.T)

# fill NaN value in movie-user matrix

## (1) sim matrix

In [900]:
item_user_matrix

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.363233
'Hellboy': The Seeds of Creation (2004),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Round Midnight (1986),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Salem's Lot (2004),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
'Til There Was You (1997),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
eXistenZ (1999),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1.29334,NaN,NaN,NaN,NaN,1.265517,NaN,NaN
xXx (2002),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.776678,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.338958,NaN,-1.969341
xXx: State of the Union (2005),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2.552484


In [901]:
from sklearn.metrics.pairwise import cosine_similarity

cos_sim_fill_zero = cosine_similarity(item_user_matrix.fillna(0)) # 평가가 유사한것이 유사도의 기준 

cos_sim_zero_df = pd.DataFrame(cos_sim_fill_zero, index=item_user_matrix.index, columns=item_user_matrix.index)
cos_sim_zero_df.head(5)

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),1.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,-0.033186,0.0,...,0.0,0.127241,-0.223349,-0.753563,0.0,0.0,-0.35184,-0.615944,0.0,0.0
'Hellboy': The Seeds of Creation (2004),0.0,1.000000,-0.298277,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.00000,0.000000,0.0,0.0
'Round Midnight (1986),0.0,-0.298277,1.000000,0.000000,0.000000,0.0,0.089156,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.00000,0.000000,0.0,0.0
'Salem's Lot (2004),0.0,0.000000,0.000000,1.000000,0.947758,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.00000,0.000000,0.0,0.0
'Til There Was You (1997),0.0,0.000000,0.000000,0.947758,1.000000,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.00000,0.000000,0.0,0.0


# (2) genre 이용 sim matrix

In [902]:
pred_ratings_use_genre.T[pred_ratings_use_genre.T.index=="'71 (2014)"]

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),<NA>,<NA>,<NA>,-0.056085,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,0.250859,<NA>,<NA>,<NA>,-0.119244,<NA>,0.311961,<NA>,0.020766


In [903]:
pred_ratings_use_genre.T

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
Toy Story (1995),<NA>,<NA>,<NA>,0.132585,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,0.052801,<NA>,<NA>,<NA>,-0.126241,<NA>,-0.210302,<NA>,0.032743
Jumanji (1995),<NA>,<NA>,<NA>,0.119899,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,-0.087448,<NA>,<NA>,<NA>,-0.194203,<NA>,-0.222777,<NA>,-0.044776
Grumpier Old Men (1995),<NA>,<NA>,<NA>,-0.084532,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,-0.125339,<NA>,<NA>,<NA>,-0.006428,<NA>,-0.298787,<NA>,0.049639
Waiting to Exhale (1995),<NA>,<NA>,<NA>,-0.074673,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,-0.117119,<NA>,<NA>,<NA>,0.055818,<NA>,-0.105509,<NA>,0.105474
Father of the Bride Part II (1995),<NA>,<NA>,<NA>,-0.034957,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,-0.094667,<NA>,<NA>,<NA>,-0.127159,<NA>,-0.368359,<NA>,0.049669
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Black Butler: Book of the Atlantic (2017),<NA>,<NA>,<NA>,0.055472,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,0.259522,<NA>,<NA>,<NA>,-0.19791,<NA>,-0.081439,<NA>,0.020914
No Game No Life: Zero (2017),<NA>,<NA>,<NA>,0.133708,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,0.285082,<NA>,<NA>,<NA>,-0.043571,<NA>,-0.169167,<NA>,0.062088
Flint (2017),<NA>,<NA>,<NA>,-0.054955,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,-0.100681,<NA>,<NA>,<NA>,0.18031,<NA>,0.281048,<NA>,0.217143


# (3) Year

In [904]:
movies_year = movies_df['year'].str.strip(to_strip="()")

In [905]:
movies_year[movies_year.apply(lambda x : len(x) > 4)]
movies_df.loc[9515, 'year'] = "(2006)"
movies_df.iloc[9515]

title          Death Note: Desu nôto (2006–2007)
movieId                                   171749
genres     [Animation, Mystery, Sci-Fi, Fantasy]
year                                      (2006)
count                                          1
Name: 9515, dtype: object

In [918]:
movies_year = movies_df['year'].str.strip(to_strip="()")

movie_ratings_df['year'] = movies_year

In [919]:
movie_ratings_df

,userId,movieId,rating,centering_rating,title,genres,year
0,1,1,4.0,-0.457947,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995
1,1,3,4.0,-0.457947,Grumpier Old Men (1995),"[Comedy, Romance]",1995
2,1,6,4.0,-0.457947,Heat (1995),"[Action, Crime, Thriller]",1995
3,1,47,5.0,0.791978,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",1995
4,1,50,5.0,0.791978,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",1995
...,...,...,...,...,...,...,...
100831,610,166534,4.0,0.363233,Split (2017),"[Drama, Horror, Thriller]",NaN
100832,610,168248,5.0,1.529520,John Wick: Chapter Two (2017),"[Action, Crime, Thriller]",NaN
100833,610,168250,5.0,1.529520,Get Out (2017),[Horror],NaN
100834,610,168252,5.0,1.529520,Logan (2017),"[Action, Sci-Fi]",NaN


In [920]:
movie_ratings_df.groupby(['userId','year'])['rating'].mean()

userId  year
1       1964    5.000000
        1967    5.000000
        1976    5.000000
        1977    5.000000
        1992    4.333333
                  ...   
64      2014    4.000000
        2015    3.750000
        2016    3.725806
        2017    3.739437
        2018    3.812500
Name: rating, Length: 1684, dtype: float64

In [923]:
user_year_matrix = movie_ratings_df.groupby(['userId','year'])['rating'].mean().unstack()

In [937]:
movie_year_matrix = pd.DataFrame(0, index=movie_ratings_df['movieId'], columns=user_year_matrix.columns)

movie_year_info = movie_ratings_df.set_index('movieId')['year'].dropna()

In [938]:
movie_year_matrix[movie_year_info]

year,1995,1995,1995,1995,1995,1995,1995,1995,1995,1995,...,2010,2013,2014,2015,2015,2017,2017,2017,2018,1991
movieId,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
47,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
50,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166534,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
168248,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
168250,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [926]:
movie_ratings_df

,userId,movieId,rating,centering_rating,title,genres,year
0,1,1,4.0,-0.457947,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1995
1,1,3,4.0,-0.457947,Grumpier Old Men (1995),"[Comedy, Romance]",1995
2,1,6,4.0,-0.457947,Heat (1995),"[Action, Crime, Thriller]",1995
3,1,47,5.0,0.791978,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",1995
4,1,50,5.0,0.791978,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",1995
...,...,...,...,...,...,...,...
100831,610,166534,4.0,0.363233,Split (2017),"[Drama, Horror, Thriller]",NaN
100832,610,168248,5.0,1.529520,John Wick: Chapter Two (2017),"[Action, Crime, Thriller]",NaN
100833,610,168250,5.0,1.529520,Get Out (2017),[Horror],NaN
100834,610,168252,5.0,1.529520,Logan (2017),"[Action, Sci-Fi]",NaN


# (4) Popular movies